# Day 38：Project 2 答辩与 AST 工程代码质量审计

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第三阶段收官之战（Day 38）！今天我们将使用 Python 标准库 `ast` 构建静态代码质量审计器 `code_review_audit.py`，对示例缺陷代码与真实迭代目录执行自动化体检（`v0.3.0-day38`）。

## 1. 环境初始化与导入 `project_iterations/day_38` 模块
定位仓库根目录并将 `project_iterations/day_38` 加入 `sys.path`，导入 `audit_python_file` 与 `audit_project_dir`。

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day38_dir = repo_root / "project_iterations" / "day_38"
if str(day38_dir) not in sys.path:
    sys.path.insert(0, str(day38_dir))

from zhishu_agent import audit_project_dir, audit_python_file

print("已加载模块路径:", day38_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_38


## 2. 实战对比：合规代码 vs. 存在安全与规范缺陷的代码
在临时目录中分别创建合规模块与高危模块（缺少文档/类型注解、包含 `eval()` 与硬编码 `sk-...` 密钥），观察 AST 审计器的精准拦截。

In [2]:
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    tmp_dir = Path(tmp)
    clean_py = tmp_dir / "clean_rag.py"
    clean_py.write_text(
        '"""合规 RAG 工具模块."""\n\n'
        'def format_citation(doc_id: str, score: float) -> str:\n'
        '    """格式化引用标签."""\n'
        '    return f"[{doc_id}] ({score:.2f})"\n',
        encoding="utf-8",
    )

    risky_py = tmp_dir / "risky_rag.py"
    risky_py.write_text(
        'API_KEY = "sk-demo-9876543210abcdef"\n\n'
        'def unsafe_calc(expr):\n'
        '    return eval(expr)\n',
        encoding="utf-8",
    )

    clean_rpt = audit_python_file(clean_py)
    risky_rpt = audit_python_file(risky_py)

    print(f"[合规文件] passed={clean_rpt.passed}, docstring={clean_rpt.docstring_coverage:.0%}, type_hint={clean_rpt.type_hint_coverage:.0%}")
    print(f"[高危文件] passed={risky_rpt.passed}, errors={risky_rpt.error_count}, warnings={risky_rpt.warning_count}")
    for issue in risky_rpt.issues:
        print(f"  - [{issue.severity.upper()}] line {issue.line} ({issue.rule}): {issue.message}")

[合规文件] passed=True, docstring=100%, type_hint=100%
[高危文件] passed=False, errors=2, warnings=3
  - [ERROR] line 1 (hardcoded-api-key): 检测到疑似硬编码 API Key (sk-demo***)，请改用环境变量注入
  - [WARNING] line 1 (missing-module-docstring): 模块缺少顶部文档字符串 (Docstring)
  - [WARNING] line 3 (missing-function-docstring): 函数 'unsafe_calc' 缺少文档字符串
  - [WARNING] line 3 (missing-type-hints): 函数 'unsafe_calc' 缺少类型注解: 参数 'expr', 返回值
  - [ERROR] line 4 (forbidden-dynamic-exec): 禁止调用危险内置函数 'eval()'


## 3. 对真实迭代目录执行全量工程审计
对 `project_iterations/day_37` 与 `project_iterations/day_38` 运行 `audit_project_dir()`，输出完整聚合审计报告。

In [3]:
for iter_name in ("day_37", "day_38"):
    target = repo_root / "project_iterations" / iter_name
    if target.is_dir():
        report = audit_project_dir(target)
        print(report.format_summary())
        print()

=== 智枢工程代码质量审计报告 (✅ PASS) ===
目标目录       : /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_37
扫描文件数     : 4
Docstring 覆盖率: 86.4%
Type Hint 覆盖率: 100.0%
阻断问题 (Error) : 0
质量告警 (Warn)  : 6
--- 问题明细 ---
[WARNING] /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_37/zhishu_agent/kb_service.py:41 (missing-function-docstring) 函数 'to_dict' 缺少文档字符串
[WARNING] /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_37/zhishu_agent/kb_service.py:55 (missing-function-docstring) 函数 'to_dict' 缺少文档字符串
[WARNING] /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_37/zhishu_agent/kb_service.py:83 (missing-function-docstring) 函数 '__init__' 缺少文档字符串
[WARNING] /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_37/zhishu_agent/kb_web_app.py:34 (missing-function-docstring) 函数 '__init__' 缺少文档字符串
[WARNING] /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_37/zhishu_agent/kb_web_app.py:40 (missing-function-docstring) 函数 '_get_conn' 缺少文档字符串
[WARNING] /home